# 01 — EHR Demographics Extraction
## NIH All of Us — Sickle Cell Disease (SCD) Analysis

| Field | Details |
|-------|---------|
| **Project** | Sickle Cell Disease Multi-Modal Analysis |
| **Dataset** | All of Us Controlled Tier Dataset v8 |
| **Description** | Extracts demographic data (age, gender, race, ethnicity) for SCD-diagnosed participants from the All of Us BigQuery CDR |
| **Input** | All of Us CDR BigQuery tables (`cb_search_all_events`, `person`) |
| **Output** | `SCD_Demographics.csv` — 476 SCD participants with demographic fields |

---

## Workflow
1. Query participant demographics from BigQuery CDR  
2. Inspect data shape and distributions  
3. Visualise gender and race distributions  
4. Save demographics to CSV for downstream notebooks

---
> ⚠️ **All of Us Compliance:** This notebook must be run inside the All of Us Researcher Workbench. No participant data is stored in this repository.

---
## ⚙️ Before You Begin

> **This notebook must be run inside the NIH All of Us Researcher Workbench.**  
> It uses BigQuery environment variables (`WORKSPACE_CDR`) that only exist inside the secure enclave.

### Requirements
- Approved All of Us Researcher Workbench access (Controlled Tier)
- Python 3 kernel selected
- Run notebooks **in order**: `01` → `02` → `03`

### How to run this notebook
1. Open your **All of Us Researcher Workbench** → your Workspace
2. Upload or clone this notebook into the Jupyter environment
3. Select **Python 3** kernel (top right)
4. Click **Kernel → Restart & Run All** to run all cells at once  
   *OR* run cells one by one using **Shift + Enter**

---

## Step 1 — Import Libraries & Query Demographics from BigQuery

**What this does:**  
Queries the All of Us BigQuery CDR `person` table to extract all SCD-diagnosed participants.

**Filters applied:**
- Age at CDR between 18–124 years
- Has whole genome variant data
- SCD diagnosis confirmed via OMOP concept IDs: `24006, 4159748, 22281, 4213628, 315523`
- Race: Black/African American, Mixed Race, or African ancestry
- Sex at birth: Female or Male only

**Expected output:** A dataframe with 476 participants and 12 demographic columns.

> ⏱️ *This query may take 1–3 minutes to run depending on CDR size.*

In [ ]:
import pandas
import os

# This query represents dataset "SCD_Demographics" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39610034_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                age_at_cdr BETWEEN 18 AND 124 
                AND NOT EXISTS (      SELECT
                    'x'      
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.death` d      
                WHERE
                    d.person_id = p.person_id ) ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (24006, 4159748, 22281, 4213628, 315523)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
            WHERE
                sex_at_birth_concept_id IN (45878463, 45880669) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
            WHERE
                race_concept_id IN (8516, 8515, 38003615) ) )"""

dataset_39610034_person_df = pandas.read_gbq(
    dataset_39610034_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_39610034_person_df.head(476)

---
## Step 2 — Inspect the Dataset

**What this does:**  
Prints the column names, data types and non-null counts to verify the query loaded correctly.

**Expected output:** 12 columns including `person_id`, `gender`, `race`, `ethnicity`, `sex_at_birth`, `date_of_birth`.

In [ ]:
dataset_39610034_person_df.info()

---
## Step 3 — Save Demographics to CSV

**What this does:**  
Saves the full demographics dataframe as `SCD_Demographics.csv` in your workspace.

> 📁 This CSV is used as the **primary input** for Notebook 02 and Notebook 03.  
> ⚠️ Do NOT commit this file to GitHub — it contains participant identifiers.

In [ ]:
# Save to CSV
dataset_39610034_person_df.to_csv("SCD_Demographics.csv", index=False)


In [ ]:
df = dataset_39610034_person_df.copy()

---
## Step 4 — Explore Basic Distributions

**What this does:**  
Prints raw value counts for gender and race to understand the cohort composition before plotting.

In [ ]:
print("\nGender distribution:")

In [ ]:
print(dataset_39610034_person_df['gender'].value_counts())

In [ ]:
print("\nRace distribution:")

In [ ]:
print(dataset_39610034_person_df['race'].value_counts())

---
## Step 5 — Visualise Gender Distribution

**What this does:**  
Creates three charts showing gender distribution:
1. Simple bar chart with participant counts
2. Bar chart with short gender codes and a legend
3. Percentage chart for Male vs Female only

**Expected output:** 3 matplotlib bar charts rendered inline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("SCD_Demographics.csv")

# Use a clean and professional plotting style
plt.style.use('seaborn-pastel')

# Check and plot gender distribution
if 'gender' in df.columns:
    gender_counts = df['gender'].value_counts()

    # Create the bar plot
    plt.figure(figsize=(6, 4))
    bars = plt.bar(gender_counts.index, gender_counts.values, color='skyblue', edgecolor='black')

    # Add counts on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 1, f'{int(height)}', ha='center', va='bottom', fontsize=10)

    # Set axis labels and title
    plt.title('Gender Distribution in SCD Cohort', fontsize=14)
    plt.xlabel('Gender', fontsize=12)
    plt.ylabel('Number of Participants', fontsize=12)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)

    # Finalize layout and show plot
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("SCD_Demographics.csv")

# Mapping full gender responses to short codes
gender_map = {
    'Male': 'M',
    'Female': 'F',
    'Prefer not to answer': 'U',         # Unknown
    'Additional Options': 'O',
    'Gender Identity': 'G',
    'PMI: Skip': 'S'
}

# Map and count
df['gender_short'] = df['gender'].map(gender_map).fillna('Other')
gender_counts = df['gender_short'].value_counts().sort_values(ascending=False)

# Plot
plt.style.use('seaborn-pastel')
plt.figure(figsize=(6, 4))
bars = plt.bar(gender_counts.index, gender_counts.values, color='steelblue', edgecolor='black')

# Annotate counts
for bar in bars:
    height = int(bar.get_height())
    plt.text(bar.get_x() + bar.get_width()/2, height + 1, str(height), ha='center', va='bottom', fontsize=10)

# Labels
plt.title('Gender Distribution in the SCD Cohort', fontsize=14)
plt.xlabel('Gender Code', fontsize=12)
plt.ylabel('Number of Participants', fontsize=12)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Legend
legend_labels = ['M: Male', 'F: Female', 'O: Other', 'U: Prefer not to answer', 'G: Gender Identity', 'S: Skipped']
plt.legend(legend_labels, title='Legend', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("SCD_Demographics.csv")

# Filter only Male and Female entries
df = df[df['gender'].isin(['Male', 'Female'])]

# Count and calculate percentage
gender_counts = df['gender'].value_counts().sort_index()  # Female, Male
gender_percent = (gender_counts / gender_counts.sum()) * 100

# Colors for each gender
colors = ['#ff7f0e', '#1f77b4']  # Female = orange, Male = blue

# Plot
plt.figure(figsize=(7, 5))
bars = plt.bar(gender_percent.index, gender_percent.values, color=colors, edgecolor='black')

# Annotate percentages
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.1f}%', 
             ha='center', va='bottom', fontsize=10)

# Labels
plt.title('Percentage Distribution of Male and Female Participants', fontsize=14)
plt.xlabel('Gender', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("SCD_Demographics.csv")

# Define gender short forms
gender_map = {
    'Male': 'M',
    'Female': 'F',
    'Prefer not to answer': 'U',
    'Additional Options': 'A',
    'Gender Identity': 'G',
    'PMI: Skip': 'S'
}

# Map gender to short codes; fill unknowns with 'O' (Other)
df['gender_short'] = df['gender'].map(gender_map).fillna('O')

# Count and calculate percentages
gender_counts = df['gender_short'].value_counts().sort_index()
gender_percent = (gender_counts / gender_counts.sum()) * 100

# Define colors
gender_colors = {
    'M': '#1f77b4',  # Male - Blue
    'F': '#ff7f0e',  # Female - Orange
    'U': '#2ca02c',  # Prefer not to answer - Green
    'A': '#d62728',  # Additional Options - Red
    'G': '#9467bd',  # Gender Identity - Purple
    'S': '#8c564b',  # Skipped - Brown
    'O': '#7f7f7f'   # Other/Unknown - Gray
}

# Plot
plt.figure(figsize=(8, 5))
bars = plt.bar(
    gender_percent.index,
    gender_percent.values,
    color=[gender_colors.get(code, 'gray') for code in gender_percent.index],
    edgecolor='black'
)

# Annotate percentages
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{height:.1f}%', 
             ha='center', va='bottom', fontsize=10)

# Labels
plt.title('Gender Distribution of All Participants (Percentage)', fontsize=14)
plt.xlabel('Gender Code', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Optional: Legend
legend_labels = {
    'M': 'Male',
    'F': 'Female',
    'U': 'Prefer not to answer',
    'A': 'Additional Options',
    'G': 'Gender Identity',
    'S': 'Skipped',
    'O': 'Other/Unknown'
}
legend_handles = [
    plt.Line2D([0], [0], color=gender_colors[code], lw=10)
    for code in gender_percent.index
]
legend_texts = [legend_labels[code] for code in gender_percent.index]

plt.legend(legend_handles, legend_texts, title='Gender', bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=10)

plt.tight_layout()
plt.show()


---
## ✅ Notebook 01 Complete

**Output file created:**
- `SCD_Demographics.csv` — 476 SCD participants with full demographic fields

**Next step:** Open and run `02_phenotypic_complications_extraction.ipynb`

---
> 💡 **Tip:** If you get a BigQuery error, check that your workspace CDR environment variable is set correctly by running `import os; print(os.environ["WORKSPACE_CDR"])` in a new cell.